In [1]:
from ucimlrepo import fetch_ucirepo 
import numpy as np
import seaborn as sb
import pandas as pd

df = pd.read_csv('../datasets/CLF/promoters.data', names=['label', 'read_name', 'sequence'])


df['sequence'] = df['sequence'].str.replace('\t', '').str.strip()


feature_cols = ["p" + str(i) for i in range(-50, 7)]


df_seq = df['sequence'].apply(lambda x: pd.Series(list(x)))
df_seq.columns = feature_cols


df = pd.concat([df[['label', 'read_name']], df_seq], axis=1)

df.head()

,label,read_name,p-50,p-49,p-48,p-47,p-46,p-45,p-44,p-43,...,p-3,p-2,p-1,p0,p1,p2,p3,p4,p5,p6
0,+,S10,t,a,c,t,a,g,c,a,...,g,g,c,t,t,g,t,c,g,t
1,+,AMPC,t,g,c,t,a,t,c,c,...,g,c,a,t,c,g,c,c,a,a
2,+,AROH,g,t,a,c,t,a,g,a,...,c,c,a,c,c,c,g,g,c,g
3,+,DEOP2,a,a,t,t,g,t,g,a,...,t,a,a,c,a,a,a,c,t,c
4,+,LEU1_TRNA,t,c,g,a,t,a,a,t,...,t,c,c,g,t,g,g,t,a,g


In [2]:
df.shape

(106, 59)

In [3]:
df.isnull().sum()

label        0
read_name    0
p-50         0
p-49         0
p-48         0
p-47         0
p-46         0
p-45         0
p-44         0
p-43         0
p-42         0
p-41         0
p-40         0
p-39         0
p-38         0
p-37         0
p-36         0
p-35         0
p-34         0
p-33         0
p-32         0
p-31         0
p-30         0
p-29         0
p-28         0
p-27         0
p-26         0
p-25         0
p-24         0
p-23         0
p-22         0
p-21         0
p-20         0
p-19         0
p-18         0
p-17         0
p-16         0
p-15         0
p-14         0
p-13         0
p-12         0
p-11         0
p-10         0
p-9          0
p-8          0
p-7          0
p-6          0
p-5          0
p-4          0
p-3          0
p-2          0
p-1          0
p0           0
p1           0
p2           0
p3           0
p4           0
p5           0
p6           0
dtype: int64

In [4]:
for c in df.columns:
    print(df[c].unique())

['+' '-']
['S10' 'AMPC' 'AROH' 'DEOP2' 'LEU1_TRNA' 'MALEFG' 'MALK' 'RECA' 'RPOB'
 'RRNAB_P1' 'RRNAB_P2' 'RRNDEX_P2' 'RRND_P1' 'RRNE_P1' 'RRNG_P1' 'RRNG_P2'
 'RRNX_P1' 'TNAA' 'TYRT' 'ARAC' 'LACI' 'MALT' 'TRP' 'TRPP2' 'THR' 'BIOB'
 'FOL' 'UVRBP1' 'UVRBP3' 'LEXA' 'PORI-L' 'SPOT42' 'M1RNA' 'GLNS' 'TUFB'
 'SUBB-E' 'STR' 'SPC' 'RPOA' 'RPLJ' 'PORI-R' 'ALAS' 'ARABAD' 'BIOA'
 'DEOP1' 'GALP2' 'HIS' 'HISJ' 'ILVGEDA' 'LACP1' 'LPP' 'TRPR' 'UVRB_P2'
 ' 867' '1169' ' 802' ' 521' ' 918' '1481' '1024' '1149' ' 313' ' 780'
 '1384' ' 507' '  39' '1203' ' 988' '1171' ' 753' ' 630' ' 660' '1216'
 ' 835' '  35' '1218' ' 668' ' 413' ' 991' ' 751' ' 850' '  93' '1108'
 ' 915' '1019' '  19' '1320' '  91' ' 217' ' 957' ' 260' ' 557' '1355'
 ' 244' ' 464' ' 296' ' 648' ' 230' '1163' '1321' ' 663' ' 799' ' 987'
 '1226' ' 794' '1442']
['t' 'g' 'a' 'c']
['a' 'g' 't' 'c']
['c' 'a' 't' 'g']
['t' 'c' 'a' 'g']
['a' 't' 'g' 'c']
['g' 't' 'a' 'c']
['c' 'g' 'a' 't']
['a' 'c' 't' 'g']
['a' 't' 'g' 'c']
['t' 'g' 'a' 'c']
['

In [5]:
df['label'] = df['label'].map({'+': 1, '-': 0})
df_features = df.drop(columns=['label', 'read_name'])

# 3. Creiamo un dizionario per mappare i nucleotidi in numeri interi
nucleotide_map = {'a': 0, 'c': 1, 'g': 2, 't': 3}

# Applichiamo la mappatura a tutte le colonne delle feature
df_features = df_features.replace(nucleotide_map)

# 4. Estraiamo le matrici NumPy finali (X e y)
X_np = df_features.values.astype(float)
y_np = df['label'].values.astype(int)



C:\Users\david\AppData\Local\Temp\ipykernel_10812\4051475554.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_features = df_features.replace(nucleotide_map)


In [6]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [7]:
# Dividi in train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)

In [8]:
# Addestra una rete (come nel paper)
mlp = MLPClassifier(
    hidden_layer_sizes=(20,),  # prova 0,5,10,20,40
    max_iter=500,
    random_state=42,
    early_stopping=False
)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)

In [9]:
print('Test Accuracy %s' % accuracy_score(y_test, y_pred))
print('Test F1-score %s' % f1_score(y_test, y_pred, average=None))
print(classification_report(y_test, y_pred))

Test Accuracy 0.8181818181818182
Test F1-score [0.8        0.83333333]
              precision    recall  f1-score   support

           0       0.89      0.73      0.80        11
           1       0.77      0.91      0.83        11

    accuracy                           0.82        22
   macro avg       0.83      0.82      0.82        22
weighted avg       0.83      0.82      0.82        22



In [10]:
# importa la classe direttamente dal file
from RuleTree.tree.TrepanClassifier import TrepanClassifier
trepan_clf = TrepanClassifier(estimator = mlp, s_min=1000, max_internal_nodes=15, random_state=42)
trepan_clf.fit(X_train, y_train)
y_pred_trepan = trepan_clf.predict(X_test)
print('Accuracy %s' % accuracy_score(y_test, y_pred_trepan))
print('F1-score %s' % f1_score(y_test, y_pred_trepan, average=None))
print(classification_report(y_test, y_pred_trepan))
fidelty = accuracy_score(y_pred, y_pred_trepan)
print("Fidelity of Trepan to the original model:", fidelty)




c:\Users\david\miniconda3\envs\trepan-dev\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accuracy 0.6363636363636364
F1-score [0.42857143 0.73333333]
              precision    recall  f1-score   support

           0       1.00      0.27      0.43        11
           1       0.58      1.00      0.73        11

    accuracy                           0.64        22
   macro avg       0.79      0.64      0.58        22
weighted avg       0.79      0.64      0.58        22

Fidelity of Trepan to the original model: 0.7272727272727273


In [11]:
attributes = feature_cols + ['label', 'read_name']
trepan_clf.print_trepan_rules(feature_names=attributes)


  REGOLE GLOBALI ESTRATTE DA TREPAN
REGOLA 1 [Nodo ID: Rl]:
  IF  (3-of-{ p-36 == 0.0, p-45 == 3.0, p-48 == 1.0 }})
  THEN Predizione = 0
  [Fedeltà: 100.00%, Copertura: 2.38%]

REGOLA 2 [Nodo ID: Rrl]:
  IF  (NOT (3-of-{ p-36 == 0.0, p-45 == 3.0, p-48 == 1.0 }})) AND 
      (1-of-{ p-36 == 0.0 }})
  THEN Predizione = 0
  [Fedeltà: 94.00%, Copertura: 21.43%]

REGOLA 3 [Nodo ID: Rrr]:
  IF  (NOT (3-of-{ p-36 == 0.0, p-45 == 3.0, p-48 == 1.0 }})) AND 
      (NOT (1-of-{ p-36 == 0.0 }}))
  THEN Predizione = 1
  [Fedeltà: 64.06%, Copertura: 76.19%]



In [12]:
rules = trepan_clf.get_rules(attributes)
trepan_clf.print_rules(rules)


|--- 3-of-{ p-36 == 0.0, p-45 == 3.0, p-48 == 1.0 }}
|    output: 0
|--- NOT (3-of-{ p-36 == 0.0, p-45 == 3.0, p-48 == 1.0 }})
|   |--- 1-of-{ p-36 == 0.0 }}
|   |    output: 0
|   |--- NOT (1-of-{ p-36 == 0.0 }})
|   |    output: 1


In [13]:
def create_mlp(hidden_units):
    if hidden_units == 0:
        return MLPClassifier(
            hidden_layer_sizes=(),
            max_iter=1000,  # aumentato
            random_state=42,
            early_stopping=False,
        )
    else:
        return MLPClassifier(
            hidden_layer_sizes=(hidden_units,),
            max_iter=1000,
            random_state=42,
            early_stopping=False,
        )

def select_best_hidden_units(X_train, y_train, hidden_units_list, cv_inner=5):
    """
    Seleziona il miglior numero di hidden unit con cross-validation interna.
    Come nel paper: prova {0,5,10,20,40} e sceglie il migliore.
    """
    X_train = np.asarray(X_train, dtype=np.float64)
    # y_train è già int, non convertire
    
    best_units = 10
    best_score = -1
    
    for units in hidden_units_list:
        mlp = create_mlp(units)
        scores = cross_val_score(mlp, X_train, y_train, cv=cv_inner, scoring='accuracy')
        mean_score = np.mean(scores)
        
        if mean_score > best_score:
            best_score = mean_score
            best_units = units
            
    return best_units, best_score

def train_and_evaluate_network(X_train, y_train, X_test, y_test, hidden_units):
    """
    Addestra una rete neurale e restituisce:
    - accuracy sul test set
    - modello addestrato
    - predizioni sul test set
    """
    mlp = create_mlp(hidden_units)
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc, mlp, y_pred

In [14]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [15]:
hidden_units_list =  [0,5,10, 20, 40] # come nel paper
n_folds = 10
random_state = 42

# Inizializza metriche
accuracies_net = []
accuracies_tree = []
fidelities = []
chosen_units = []

skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)

print("\n" + "="*60)
print("INIZIO 10-FOLD CROSS-VALIDATION")
print("="*60 + "\n")

# ============================================================
# 6. 10-FOLD CROSS-VALIDATION
# ============================================================
for fold, (train_idx, test_idx) in enumerate(skf.split(X_np, y_np), 1):
    print(f"Fold {fold}/{n_folds}")
    
    X_train = X_np[train_idx]  
    y_train = y_np[train_idx]
    X_test = X_np[test_idx]
    y_test = y_np[test_idx]

    # Seleziona il miglior numero di hidden unit
    best_units, best_cv_score = select_best_hidden_units(
        X_train, y_train, hidden_units_list, cv_inner=5
    )
    chosen_units.append(best_units)
    print(f"  → Migliori hidden units: {best_units} (CV score interno: {best_cv_score:.3f})")
    
    # Addestra la rete finale
    net_acc, network, y_pred_net = train_and_evaluate_network(
        X_train, y_train, X_test, y_test, best_units
    )
    accuracies_net.append(net_acc)
    print(f"  → Accuratezza rete: {net_acc:.3f}")
    
    # ============================================================
    # 7. TREPAN - Estrazione dell'albero
    # ============================================================
    
    # Se TREPAN è importato, usalo
    trepan_clf = TrepanClassifier(estimator=network, s_min=500, max_leaf_nodes=16, random_state=42)
    trepan_clf.fit(X_train, y_train)
    y_pred_trepan = trepan_clf.predict(X_test)
    
    
    
    tree_acc = accuracy_score(y_test, y_pred_trepan)
    fidelity = accuracy_score(y_pred_net, y_pred_trepan)
    
    accuracies_tree.append(tree_acc)
    fidelities.append(fidelity)
    
    print(f"  → Accuratezza albero: {tree_acc:.3f}")
    print(f"  → Fedeltà albero-rete: {fidelity:.3f}")
    print()

# ============================================================
# 8. RISULTATI FINALI
# ============================================================
print("="*60)
print("RISULTATI FINALI (10-fold CV)")
print("="*60)
print(f"Accuratezza rete:     {np.mean(accuracies_net):.3f} ± {np.std(accuracies_net):.3f}")
print(f"Accuratezza albero:   {np.mean(accuracies_tree):.3f} ± {np.std(accuracies_tree):.3f}")
print(f"Fedeltà albero-rete:  {np.mean(fidelities):.3f} ± {np.std(fidelities):.3f}")
print(f"Hidden units più scelte: {pd.Series(chosen_units).value_counts().to_dict()}")


INIZIO 10-FOLD CROSS-VALIDATION

Fold 1/10
  → Migliori hidden units: 5 (CV score interno: 0.716)
  → Accuratezza rete: 0.818
  → Accuratezza albero: 0.818
  → Fedeltà albero-rete: 0.818

Fold 2/10
  → Migliori hidden units: 40 (CV score interno: 0.779)
  → Accuratezza rete: 0.818
  → Accuratezza albero: 0.818
  → Fedeltà albero-rete: 1.000

Fold 3/10
  → Migliori hidden units: 40 (CV score interno: 0.737)
  → Accuratezza rete: 0.818
  → Accuratezza albero: 0.909
  → Fedeltà albero-rete: 0.727

Fold 4/10
  → Migliori hidden units: 40 (CV score interno: 0.789)
  → Accuratezza rete: 0.818
  → Accuratezza albero: 0.818
  → Fedeltà albero-rete: 0.818

Fold 5/10
  → Migliori hidden units: 40 (CV score interno: 0.779)
  → Accuratezza rete: 0.818
  → Accuratezza albero: 0.818
  → Fedeltà albero-rete: 0.818

Fold 6/10
  → Migliori hidden units: 40 (CV score interno: 0.747)
  → Accuratezza rete: 0.909
  → Accuratezza albero: 0.818
  → Fedeltà albero-rete: 0.727

Fold 7/10
  → Migliori hidden u

In [16]:
import numpy as np
from sklearn.metrics import accuracy_score
from RuleTree.tree.TrepanClassifier import TrepanClassifier

# 1. Definisci i seed per lo stress test
seeds_da_testare = [10, 42, 111, 123, 999, 11, 86, 78, 123, 22, 52, 66, 88, 101, 222]

# 2. Inizializza le liste per raccogliere le metriche
varianza_accuracy = []
varianza_fidelity = []
varianza_foglie = []

print("Calcolo delle predizioni dell'oracolo (fisse per tutti i seed)...")
# L'oracolo è il modello mlp già addestrato
y_pred_oracolo = mlp.predict(X_test)

for seed in seeds_da_testare:
    print(f"\nAddestramento albero con Seed: {seed}")
    
    # Inizializza TrepanClassifier con il seed corrente
    trepan_clf = TrepanClassifier(
        estimator=mlp, 
        s_min=500, 
        max_leaf_nodes=16, 
        random_state=seed,
    )
    
    # Esegui il fit sui dati di training
    trepan_clf.fit(X_train, y_train)
    
    # Ottieni le predizioni dell'albero su X_test_scaled
    y_pred_trepan = trepan_clf.predict(X_test)
    
    # Calcola l'Accuracy
    acc = accuracy_score(y_test, y_pred_trepan)
    
    # Calcola la Fidelity
    fid = accuracy_score(y_pred_oracolo, y_pred_trepan)
    
    # Calcola il numero di foglie generate
    foglie = len(trepan_clf.get_leaf_nodes())
    
    # Aggiungi i valori alle liste
    varianza_accuracy.append(acc)
    varianza_fidelity.append(fid)
    varianza_foglie.append(foglie)

print("\n" + "="*50)
print(" RISULTATI TEST DI VARIANZA DEI RANDOM SEED")
print("="*50)

print(f"Accuracy : Media {np.mean(varianza_accuracy):.4f} ± Std {np.std(varianza_accuracy):.4f}")
print(f"Fidelity : Media {np.mean(varianza_fidelity):.4f} ± Std {np.std(varianza_fidelity):.4f}")
print(f"Foglie   : Media {np.mean(varianza_foglie):.2f} ± Std {np.std(varianza_foglie):.2f}")

Calcolo delle predizioni dell'oracolo (fisse per tutti i seed)...

Addestramento albero con Seed: 10

Addestramento albero con Seed: 42

Addestramento albero con Seed: 111

Addestramento albero con Seed: 123

Addestramento albero con Seed: 999

Addestramento albero con Seed: 11

Addestramento albero con Seed: 86

Addestramento albero con Seed: 78

Addestramento albero con Seed: 123

Addestramento albero con Seed: 22

Addestramento albero con Seed: 52

Addestramento albero con Seed: 66

Addestramento albero con Seed: 88

Addestramento albero con Seed: 101

Addestramento albero con Seed: 222

 RISULTATI TEST DI VARIANZA DEI RANDOM SEED
Accuracy : Media 0.7067 ± Std 0.0772
Fidelity : Media 0.7067 ± Std 0.0772
Foglie   : Media 12.13 ± Std 4.18
